# Hybrid Approach Sampling for Instance Segmentation

This notebook implements a four-phase hybrid sampling strategy to address class imbalance while preserving geologically meaningful data compositions.

## Algorithm Overview

**Phase 1: Intelligent Undersampling** - Filter dataset by composition, retaining images with minority classes and subsampling pure-majority images.

**Phase 2: Clean Background Preparation** - Generate masks and inpainted backgrounds free of majority class artifacts.

**Phase 3: Realistic Copy-Paste Oversampling** - Synthesize new images by pasting minority instances onto cleaned backgrounds with controlled overlap and occlusion.

**Phase 4: Hybrid Assembly and Validation** - Combine undersampled and synthetic datasets, validate class distribution and visual quality.

## Target Distribution (Post-Hybrid)
- Silt: 20-25% (reduced from 33%)
- Coal: 15-18%
- Shalestone: 15-18%
- Limestone, Sandstone, Quartz: 12-15% each (increased from 8-10%)

In [ ]:
import os
import cv2
import json
import random
import shutil
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

## Phase 1: Intelligent Undersampling

In [ ]:
INPUT_DATASET_DIR = "/path/to/YOLO_dataset"
OUTPUT_BASE_DIR = "/path/to/output"
DATASET_FORMAT = "YOLO"

CLASS_MAP = {0: "Silt", 1: "Sandstone", 2: "Limestone", 3: "Coal", 4: "Shalestone", 5: "Quartz"}
MINORITY_CLASSES = [1, 2, 5]
MAJORITY_CLASSES = [0]
MAJOR_MINORITY_CLASSES = [3, 4]

# Resampling Targets Configuration
UNDERSAMPLING_SILT = {
    "delete_images_100_silt": True,
    "target_instances": 3500,
    "max_delete_images": 500
}

OVERSAMPLING_MINORITY = {
    1: {"target_instances": 2000, "max_copy_per_image": 3},
    2: {"target_instances": 2200, "max_copy_per_image": 5},
    5: {"target_instances": 2200, "max_copy_per_image": 5},
}

# Augmentation Control
IS_USING_AUG = True

BACKGROUND_CONFIG = {
    "inpaint_method": "telea",
    "inpaint_radius": 3,
    "gaussian_blur_kernel": (5, 5)
}

SYNTHETIC_CONFIG = {
    "max_area_occupancy": 0.35,
    "overlap_probability": 0.25,
    "scale_range": (0.85, 1.15),
    "flip_probability": 0.5,
    "brightness_jitter": 0.15,
    "edge_feather_kernel": (3, 3),
    "min_object_area": 100
}

from pathlib import Path
for subdir in ["phase1_undersampled", "phase2_backgrounds", "phase3_synthetic", "phase4_hybrid"]:
    Path(OUTPUT_BASE_DIR).joinpath(subdir, "train", "images").mkdir(parents=True, exist_ok=True)
    Path(OUTPUT_BASE_DIR).joinpath(subdir, "train", "labels").mkdir(parents=True, exist_ok=True)

## Utility Functions

In [ ]:
def parse_yolo_label(label_path, img_width, img_height):
    """Parse YOLO label file and return list of (class_id, polygon) tuples."""
    annotations = []
    if not os.path.exists(label_path):
        return annotations
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            class_id = int(parts[0])
            coords = [float(p) for p in parts[1:]]
            polygon = np.array([(coords[i]*img_width, coords[i+1]*img_height) 
                               for i in range(0, len(coords)-1, 2)], dtype=np.int32)
            annotations.append((class_id, polygon))
    return annotations

def compute_image_composition(annotations):
    """Return (total_count, class_counts_dict, silt_ratio)."""
    class_counts = {}
    for class_id, _ in annotations:
        class_counts[class_id] = class_counts.get(class_id, 0) + 1
    total = sum(class_counts.values())
    silt_count = class_counts.get(0, 0)
    silt_ratio = silt_count / total if total > 0 else 0.0
    return total, class_counts, silt_ratio

def extract_object_rgba(image, polygon):
    """Extract object RGBA with alpha channel as mask."""
    x, y, w, h = cv2.boundingRect(polygon)
    x, y = max(0, x), max(0, y)
    w, h = min(w, image.shape[1]-x), min(h, image.shape[0]-y)
    
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [polygon], 255)
    
    obj_crop = image[y:y+h, x:x+w].copy()
    obj_mask = mask[y:y+h, x:x+w].copy()
    
    b, g, r = cv2.split(obj_crop)
    obj_rgba = cv2.merge([b, g, r, obj_mask])
    return obj_rgba, (x, y)

def polygon_to_yolo_string(contour, img_w, img_h):
    """Convert contour to normalized YOLO polygon string."""
    if len(contour) < 3:
        return None
    contour = contour.squeeze()
    if contour.ndim == 1 or len(contour) < 3:
        return None
    norm_pts = []
    for pt in contour:
        x_norm = np.clip(pt[0] / img_w, 0.0, 1.0)
        y_norm = np.clip(pt[1] / img_h, 0.0, 1.0)
        norm_pts.extend([f"{x_norm:.6f}", f"{y_norm:.6f}"])
    return " ".join(norm_pts) if len(norm_pts) >= 6 else None

## Phase 1: Filtering and Subsampling

In [2]:
def execute_phase1_undersampling(input_dir, output_dir):
    """Apply targeted undersampling for Silt based on configuration."""
    labels_dir = os.path.join(input_dir, "train", "labels")
    images_dir = os.path.join(input_dir, "train", "images")
    
    keep_list = []
    removal_candidates = []
    current_silt_instances = 0
    
    for label_file in tqdm(list(Path(labels_dir).glob("*.txt")), desc="Phase 1: Filtering"):
        img_name = label_file.stem + ".jpg"
        img_path = os.path.join(images_dir, img_name)
        if not os.path.exists(img_path):
            img_name = label_file.stem + ".png"
            img_path = os.path.join(images_dir, img_name)
        if not os.path.exists(img_path):
            continue
        
        image = cv2.imread(img_path)
        if image is None:
            continue
        
        h, w = image.shape[:2]
        annotations = parse_yolo_label(str(label_file), w, h)
        _, class_counts, silt_ratio = compute_image_composition(annotations)
        
        minority_count = sum(1 for cid, _ in annotations if cid in MINORITY_CLASSES)
        silt_count = class_counts.get(0, 0)
        
        # Decision logic
        if minority_count > 0:
            keep_list.append((img_path, str(label_file), img_name))
            current_silt_instances += silt_count
        elif UNDERSAMPLING_SILT["delete_images_100_silt"] and silt_ratio == 1.0:
            removal_candidates.append((img_path, str(label_file), img_name, silt_count))
        else:
            keep_list.append((img_path, str(label_file), img_name))
            current_silt_instances += silt_count
            
    print(f"[Phase 1] Initial KEEP: {len(keep_list)}, REMOVAL CANDIDATES: {len(removal_candidates)}")
    
    # Process removal candidates to hit target
    random.shuffle(removal_candidates)
    deleted_count = 0
    
    for item in removal_candidates:
        if deleted_count >= UNDERSAMPLING_SILT["max_delete_images"]:
            keep_list.append((item[0], item[1], item[2]))
            current_silt_instances += item[3]
        elif current_silt_instances > UNDERSAMPLING_SILT["target_instances"]:
            deleted_count += 1
        else:
            keep_list.append((item[0], item[1], item[2]))
            current_silt_instances += item[3]
            
    print(f"[Phase 1] Deleted {deleted_count} images. Final Silt instances: {current_silt_instances}")
    
    for img_path, lbl_path, img_name in tqdm(keep_list, desc="Phase 1: Writing"):
        shutil.copy2(img_path, os.path.join(output_dir, "train", "images", img_name))
        lbl_name = Path(img_name).stem + ".txt"
        shutil.copy2(lbl_path, os.path.join(output_dir, "train", "labels", lbl_name))
    
    return keep_list

## Phase 2: Clean Background Preparation

In [3]:
def execute_phase2_background_preparation(dataset_dir):
    """Generate clean backgrounds by classifying and inpainting Silt regions."""
    labels_dir = os.path.join(dataset_dir, "train", "labels")
    images_dir = os.path.join(dataset_dir, "train", "images")
    
    pool_a = []
    pool_b = []
    
    for label_file in tqdm(list(Path(labels_dir).glob("*.txt")), desc="Phase 2: Pool Classification"):
        img_name = label_file.stem + ".jpg"
        img_path = os.path.join(images_dir, img_name)
        if not os.path.exists(img_path):
            img_name = label_file.stem + ".png"
            img_path = os.path.join(images_dir, img_name)
        if not os.path.exists(img_path):
            continue
        
        image = cv2.imread(img_path)
        if image is None:
            continue
        
        h, w = image.shape[:2]
        annotations = parse_yolo_label(str(label_file), w, h)
        silt_count = sum(1 for cid, _ in annotations if cid == 0)
        
        if silt_count == 0:
            pool_a.append((img_path, img_name))
        else:
            pool_b.append((img_path, str(label_file), img_name, annotations, h, w))
    
    print(f"[Phase 2] Pool A (no Silt): {len(pool_a)}, Pool B (with Silt): {len(pool_b)}")
    
    inpainted_backgrounds = []
    for img_path, lbl_path, img_name, annotations, h, w in tqdm(pool_b, desc="Phase 2: Inpainting"):
        image = cv2.imread(img_path)
        void_mask = np.ones((h, w), dtype=np.uint8) * 255
        
        for class_id, polygon in annotations:
            if class_id == 0:
                cv2.fillPoly(void_mask, [polygon], 0)
        
        inpainted = cv2.inpaint(image, (255 - void_mask), 
                                BACKGROUND_CONFIG["inpaint_radius"], 
                                cv2.INPAINT_TELEA)
        
        silt_residual = cv2.countNonZero(255 - void_mask)
        if silt_residual < 0.05 * (h * w):
            inpainted_backgrounds.append((inpainted, img_name))
    
    combined_bg_pool = [(cv2.imread(p), n) for p, n in pool_a] + inpainted_backgrounds
    print(f"[Phase 2] Final background pool size: {len(combined_bg_pool)}")
    
    return combined_bg_pool

## Phase 3: Realistic Copy-Paste Oversampling

In [4]:
def execute_phase3_synthetic_generation(phase1_dir, background_pool, output_dir):
    """Synthesize images by copy-pasting objects subject to OVERSAMPLING_MINORITY targets."""
    if not background_pool:
        print("[Phase 3] Background pool empty, skipping synthesis")
        return
    
    # 1. Build object bank
    object_bank = {cid: [] for cid in OVERSAMPLING_MINORITY.keys()}
    labels_dir = os.path.join(phase1_dir, "train", "labels")
    images_dir = os.path.join(phase1_dir, "train", "images")
    
    current_counts = {cid: 0 for cid in OVERSAMPLING_MINORITY.keys()}
    
    for label_file in tqdm(list(Path(labels_dir).glob("*.txt")), desc="Phase 3: Object Bank"):
        img_name = label_file.stem + ".jpg"
        img_path = os.path.join(images_dir, img_name)
        if not os.path.exists(img_path):
            img_name = label_file.stem + ".png"
            img_path = os.path.join(images_dir, img_name)
        if not os.path.exists(img_path):
            continue
            
        with open(label_file, 'r') as f:
            for line in f:
                cid = int(line.strip().split()[0])
                if cid in current_counts:
                    current_counts[cid] += 1
        
        image = cv2.imread(img_path)
        if image is None:
            continue
        h, w = image.shape[:2]
        
        annotations = parse_yolo_label(str(label_file), w, h)
        for class_id, polygon in annotations:
            if class_id in OVERSAMPLING_MINORITY:
                area = cv2.contourArea(polygon)
                if area >= SYNTHETIC_CONFIG["min_object_area"] and area < 0.4 * (h * w):
                    obj_rgba, offset = extract_object_rgba(image, polygon)
                    object_bank[class_id].append({"rgba": obj_rgba, "offset": offset})
    
    print(f"[Phase 3] Initial minority counts: {current_counts}")
    
    # Check if targets are already reached
    classes_to_augment = [cid for cid in OVERSAMPLING_MINORITY 
                          if current_counts[cid] < OVERSAMPLING_MINORITY[cid]["target_instances"]]
    
    if not classes_to_augment:
        print("[Phase 3] All targets reached. Skipping synthesis.")
        return
        
    syn_idx = 0
    while classes_to_augment:
        bg_img, _ = random.choice(background_pool)
        bg_h, bg_w = bg_img.shape[:2]
        
        instance_map = np.zeros((bg_h, bg_w), dtype=np.int32)
        class_map = {}
        inst_id = 1
        area_occupied = 0
        max_area = bg_h * bg_w * SYNTHETIC_CONFIG["max_area_occupancy"]
        
        for cid in list(classes_to_augment):
            if current_counts[cid] >= OVERSAMPLING_MINORITY[cid]["target_instances"]:
                classes_to_augment.remove(cid)
                continue
                
            num_paste = random.randint(1, OVERSAMPLING_MINORITY[cid]["max_copy_per_image"])
            if not object_bank[cid]: continue
            
            for _ in range(num_paste):
                if current_counts[cid] >= OVERSAMPLING_MINORITY[cid]["target_instances"]:
                    break
                    
                obj_data = random.choice(object_bank[cid])
                aug_rgba = augment_object(obj_data["rgba"]) if IS_USING_AUG else obj_data["rgba"].copy()
                obj_h, obj_w = aug_rgba.shape[:2]
                
                if obj_h >= bg_h or obj_w >= bg_w: continue
                
                obj_area = cv2.countNonZero(aug_rgba[:, :, 3])
                if area_occupied + obj_area > max_area: break
                
                max_y, max_x = max(0, bg_h - obj_h), max(0, bg_w - obj_w)
                paste_y, paste_x = random.randint(0, max_y), random.randint(0, max_x)
                
                overlap_ratio = compute_overlap_ratio(instance_map, paste_y, paste_x, obj_h, obj_w)
                if overlap_ratio > 0.5 and random.random() > SYNTHETIC_CONFIG["overlap_probability"]:
                    continue
                
                blend_object_onto_background(bg_img, aug_rgba, paste_y, paste_x, 
                                            instance_map, inst_id, cid, class_map)
                area_occupied += obj_area
                inst_id += 1
                current_counts[cid] += 1
                
        if class_map:
            output_img_path = os.path.join(output_dir, "train", "images", f"synth_{syn_idx:05d}.jpg")
            cv2.imwrite(output_img_path, bg_img)
            
            output_lbl_path = os.path.join(output_dir, "train", "labels", f"synth_{syn_idx:05d}.txt")
            with open(output_lbl_path, 'w') as f:
                for iid, class_id in class_map.items():
                    inst_mask = (instance_map == iid).astype(np.uint8) * 255
                    if cv2.countNonZero(inst_mask) < 50:
                        continue
                    contours, _ = cv2.findContours(inst_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    for contour in contours:
                        yolo_str = polygon_to_yolo_string(contour, bg_w, bg_h)
                        if yolo_str:
                            f.write(f"{class_id} {yolo_str}\n")
            syn_idx += 1
            if syn_idx % 50 == 0:
                print(f"[Phase 3] Generated {syn_idx} images. Current counts: {current_counts}")

def augment_object(obj_rgba):
    """Apply augmentation: scale, flip, brightness, feathering."""
    scale = random.uniform(*SYNTHETIC_CONFIG["scale_range"])
    h, w = obj_rgba.shape[:2]
    new_w, new_h = max(1, int(w*scale)), max(1, int(h*scale))
    scaled = cv2.resize(obj_rgba, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    
    if random.random() < SYNTHETIC_CONFIG["flip_probability"]:
        scaled = cv2.flip(scaled, 1)
    
    b, g, r, alpha = cv2.split(scaled)
    rgb = cv2.merge([b, g, r])
    hsv = cv2.cvtColor(rgb, cv2.COLOR_BGR2HSV).astype(np.float32)
    brightness_factor = 1.0 + random.uniform(-SYNTHETIC_CONFIG["brightness_jitter"], 
                                             SYNTHETIC_CONFIG["brightness_jitter"])
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * brightness_factor, 0, 255)
    hsv = hsv.astype(np.uint8)
    rgb_aug = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    
    alpha_feather = cv2.GaussianBlur(alpha, SYNTHETIC_CONFIG["edge_feather_kernel"], 0)
    b, g, r = cv2.split(rgb_aug)
    return cv2.merge([b, g, r, alpha_feather])

def compute_overlap_ratio(instance_map, y, x, h, w):
    """Compute overlap ratio between new placement and existing instances."""
    roi = instance_map[y:y+h, x:x+w]
    overlap = np.sum(roi > 0)
    total = h * w
    return overlap / total if total > 0 else 0.0

def blend_object_onto_background(bg_img, obj_rgba, y, x, instance_map, inst_id, class_id, class_map):
    """Blend object onto background with alpha blending and instance tracking."""
    obj_h, obj_w = obj_rgba.shape[:2]
    b, g, r, alpha = cv2.split(obj_rgba)
    obj_rgb = cv2.merge([b, g, r])
    
    bg_roi = bg_img[y:y+obj_h, x:x+obj_w]
    alpha_norm = alpha.astype(np.float32) / 255.0
    
    for c in range(3):
        bg_roi[:, :, c] = (1.0 - alpha_norm) * bg_roi[:, :, c].astype(np.float32) + \
                          alpha_norm * obj_rgb[:, :, c].astype(np.float32)
    bg_img[y:y+obj_h, x:x+obj_w] = bg_roi.astype(np.uint8)
    
    binary_mask = (alpha > 128).astype(np.uint8)
    instance_map[y:y+obj_h, x:x+obj_w][binary_mask == 1] = inst_id
    class_map[inst_id] = class_id

## Phase 4: Assembly and Validation

In [5]:
def generate_hybrid_dataset(input_dir, output_dir):
    """Orchestrate the hybrid resampling pipeline."""
    print("=== Phase 0: Setup ===")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        
    os.makedirs(os.path.join(output_dir, "train", "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "train", "labels"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "val", "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "val", "labels"), exist_ok=True)
    
    # Also need a temp directory for Phase 1 output
    temp_p1_dir = os.path.join(output_dir, "temp_phase1")
    os.makedirs(os.path.join(temp_p1_dir, "train", "images"), exist_ok=True)
    os.makedirs(os.path.join(temp_p1_dir, "train", "labels"), exist_ok=True)
    
    print("\n=== Phase 1: Context-Aware Undersampling ===")
    keep_list = execute_phase1_undersampling(input_dir, temp_p1_dir)
    
    print("\n=== Phase 2: Background Preparation ===")
    background_pool = execute_phase2_background_preparation(temp_p1_dir)
    
    print("\n=== Phase 3: Synthetic Minority Generation ===")
    execute_phase3_synthetic_generation(temp_p1_dir, background_pool, output_dir)
    
    print("\n=== Phase 4: Consolidate Final Dataset ===")
    # Copy Phase 1 survivors to final output
    for img_path, lbl_path, img_name in tqdm(keep_list, desc="Phase 4: Copying survivors"):
        shutil.copy2(img_path, os.path.join(output_dir, "train", "images", img_name))
        lbl_name = Path(lbl_path).name
        shutil.copy2(lbl_path, os.path.join(output_dir, "train", "labels", lbl_name))
        
    # Copy val split blindly from input_dir
    val_images_in = os.path.join(input_dir, "val", "images")
    val_labels_in = os.path.join(input_dir, "val", "labels")
    if os.path.exists(val_images_in):
        for img in tqdm(list(Path(val_images_in).glob("*.*")), desc="Phase 4: Copying Val Images"):
            shutil.copy2(str(img), os.path.join(output_dir, "val", "images", img.name))
    if os.path.exists(val_labels_in):
        for lbl in tqdm(list(Path(val_labels_in).glob("*.txt")), desc="Phase 4: Copying Val Labels"):
            shutil.copy2(str(lbl), os.path.join(output_dir, "val", "labels", lbl.name))
            
    # Cleanup temp dir
    if os.path.exists(temp_p1_dir):
        shutil.rmtree(temp_p1_dir)
    print("\n=== Pipeline Complete ===")
    print(f"Final dataset path: {output_dir}")

# Execution Configuration Setup
INPUT_DATASET_DIR = "../sam-annotation/sam_output"
OUTPUT_DATASET_DIR = "../sam-annotation/sam_output_hybrid_balanced"

# Run Pipeline
# generate_hybrid_dataset(INPUT_DATASET_DIR, OUTPUT_DATASET_DIR)